
# 16 — Spatial Evidence Synthesis for ISEF

This notebook is designed to strengthen the project **without changing the core pipeline**.  
It answers three judge-critical questions using **held-out site results**:

1. **Is performance truly above random across independent sites?**
2. **Is the California pattern geographically structured, or just noise?**
3. **Does the full early-warning model outperform a simpler baseline, and where?**

Why this is a strong next step:
- it uses **post hoc synthesis**, not new feature tuning
- it is easy to explain on a board
- it adds a **California map**, a **forest plot**, and **uncertainty-aware statistics**
- it can optionally extend to **calibration / actionability** if you paste quarter-level predictions later

The notebook runs out of the box with a default demo table based on your current blind-site summary.  
Replace the CSV blocks with your latest verified results whenever you are ready.


In [1]:

# Core imports and settings
from pathlib import Path
from io import StringIO
import textwrap
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from scipy.stats import mannwhitneyu, binomtest, wilcoxon
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.calibration import calibration_curve

warnings.filterwarnings("ignore")

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 1.2,
    "figure.dpi": 140,
})

CENTRAL_LAT_MIN = 36.5
CENTRAL_LAT_MAX = 38.5

N_PERM = 3000
N_BOOT = 2000
RANDOM_SEED = 42

OUTPUT_DIR = Path("16_spatial_evidence_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Output directory: {OUTPUT_DIR.resolve()}")
print(f"Permutation iterations: {N_PERM:,}")
print(f"Bootstrap iterations:   {N_BOOT:,}")


ModuleNotFoundError: No module named 'sklearn'


## 1) Site metadata and California coastline

You normally should not change this cell unless you rename sites.
It includes:
- canonical site coordinates
- a few name aliases from your earlier notebooks
- a lightweight California coastline so the map has **no geospatial dependency**


In [3]:

SITE_META = pd.DataFrame([
    ("Crescent City",   41.75, -124.00),
    ("Cape Mendocino",  40.45, -124.30),
    ("Bodega Bay",      38.40, -123.15),
    ("Point Reyes",     38.00, -122.85),
    ("Half Moon Bay",   37.50, -122.45),
    ("Santa Cruz",      37.00, -122.10),
    ("Point Sur",       36.40, -121.70),
    ("Cambria",         35.60, -121.00),
    ("Morro Bay",       35.35, -120.75),
    ("Point Conception",34.50, -120.45),
    ("Santa Barbara",   34.35, -119.80),
    ("Ventura",         34.25, -119.20),
    ("Palos Verdes",    33.75, -118.30),
    ("Laguna Beach",    33.55, -117.65),
    ("San Diego",       32.80, -117.20),
    ("NorCal Region",   39.00, -123.75),
    ("MidCal Region",   36.75, -122.25),
    ("BigSur Region",   35.40, -121.50),
    ("SoCal Region",    34.30, -119.90),
], columns=["canonical_site", "lat_meta", "lon_meta"])

SITE_ALIASES = {
    "NorCal (train)": "NorCal Region",
    "MidCal (train)": "MidCal Region",
    "Big Sur (train)": "BigSur Region",
    "BigSur (train)": "BigSur Region",
    "SoCal (train)": "SoCal Region",
    "NorCal (region)": "NorCal Region",
    "MidCal (region)": "MidCal Region",
    "Big Sur (region)": "BigSur Region",
    "BigSur (region)": "BigSur Region",
    "SoCal (region)": "SoCal Region",
}

CA_COAST_LON = [
    -124.21, -124.35, -124.56, -124.61, -124.51, -124.41, -124.21,
    -124.07, -123.82, -123.69, -123.44, -123.11, -122.88, -122.61,
    -122.47, -122.39, -122.17, -121.90, -121.55, -121.29, -120.89,
    -120.67, -120.47, -120.17, -119.87, -119.53, -119.23, -118.98,
    -118.56, -118.29, -118.00, -117.67, -117.42, -117.25, -117.12
]
CA_COAST_LAT = [
    41.98, 41.72, 41.31, 40.83, 40.44, 40.04, 39.73,
    39.36, 38.97, 38.54, 38.24, 37.95, 37.75, 37.52,
    37.21, 36.97, 36.60, 36.27, 35.82, 35.42, 35.15,
    34.90, 34.52, 34.26, 34.05, 33.89, 33.72, 33.48,
    33.22, 32.98, 32.76, 32.63, 32.54, 32.53, 32.54
]

print(f"{len(SITE_META)} canonical sites/regions loaded.")


19 canonical sites/regions loaded.



## 2) Paste your result tables here

### Required table: `SITE_SUMMARY_CSV`
Minimum columns:
- `site`
- `auc`

Strongly recommended:
- `ci_lo`, `ci_hi`
- `n_onset`
- `region_type`
- `auc_baseline` or a separate baseline table

### Optional table: `BASELINE_SITE_CSV`
Use this if you have a **site-level baseline AUC** to compare against the full model.
Expected columns:
- `site`
- `auc_baseline`

### Optional table: `PREDICTIONS_CSV`
Use this later if you want the notebook to run **calibration + horizon-aware utility**.
Expected columns:
- `site`
- `date`
- `onset`
- `prob_full`
- optional: `prob_baseline`

You can also load from external CSV files by setting the path variables.


In [ ]:

# --- Option A: paste CSV text directly ---
SITE_SUMMARY_CSV = """site,auc,ci_lo,ci_hi,n_onset,region_type
NorCal (train),0.6,0.44,0.76,5,train
MidCal (train),0.81,0.63,0.98,4,train
Big Sur (train),0.742,0.445,0.972,4,train
SoCal (train),0.55,0.2,0.9,2,train
Crescent City,0.454,0.149,0.882,5,new
Cape Mendocino,0.569,0.278,0.891,5,new
Bodega Bay,0.659,0.507,0.803,4,new
Point Reyes,0.532,0.316,0.74,4,new
Half Moon Bay,0.651,0.413,1.0,4,new
Santa Cruz,0.8,0.648,1.0,4,new
Point Sur,0.449,0.187,0.703,6,new
Cambria,0.538,0.161,0.945,5,new
Morro Bay,0.509,0.077,0.954,5,new
Point Conception,0.606,0.201,0.889,3,new
Santa Barbara,0.565,0.362,0.772,5,new
Ventura,0.456,0.203,0.702,8,new
Palos Verdes,0.484,0.005,0.976,4,new
Laguna Beach,0.606,0.122,0.929,4,new
San Diego,0.419,0.058,0.851,4,new"""

BASELINE_SITE_CSV = """
site,auc_baseline
"""

PREDICTIONS_CSV = """
site,date,onset,prob_full,prob_baseline
"""

# --- Option B: load from files instead of pasted CSV text ---
SITE_SUMMARY_PATH = None      # example: "site_summary.csv"
BASELINE_SITE_PATH = None     # example: "site_baseline.csv"
PREDICTIONS_PATH = None       # example: "oof_predictions.csv"

print("Ready for data input.")


In [ ]:

def _read_csv_from_text_or_path(csv_text, path, label):
    if path is not None:
        path = Path(path)
        if not path.exists():
            raise FileNotFoundError(f"{label}: file not found -> {path}")
        return pd.read_csv(path)

    csv_text = csv_text.strip()
    if not csv_text:
        return pd.DataFrame()

    # guard against header-only placeholders
    lines = [line for line in csv_text.splitlines() if line.strip()]
    if len(lines) <= 1:
        return pd.DataFrame()

    return pd.read_csv(StringIO(csv_text))

def prepare_site_summary(df):
    if df.empty:
        raise ValueError("SITE_SUMMARY_CSV is empty. Paste at least site and auc.")

    required = {"site", "auc"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"SITE_SUMMARY_CSV missing required columns: {sorted(missing)}")

    out = df.copy()
    out["site"] = out["site"].astype(str).str.strip()
    out["canonical_site"] = out["site"].replace(SITE_ALIASES)

    out = out.merge(SITE_META, on="canonical_site", how="left")

    if "lat" in out.columns:
        out["lat"] = pd.to_numeric(out["lat"], errors="coerce").fillna(out["lat_meta"])
    else:
        out["lat"] = out["lat_meta"]

    if "lon" in out.columns:
        out["lon"] = pd.to_numeric(out["lon"], errors="coerce").fillna(out["lon_meta"])
    else:
        out["lon"] = out["lon_meta"]

    unmatched = out.loc[out["lat"].isna() | out["lon"].isna(), "site"].unique().tolist()
    if unmatched:
        raise ValueError(
            "These site names could not be matched to metadata. "
            f"Either add lat/lon columns or update SITE_ALIASES: {unmatched}"
        )

    numeric_cols = [
        "auc", "ci_lo", "ci_hi", "n_onset", "auc_baseline", "delta_auc"
    ]
    for col in numeric_cols:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    if "region_type" not in out.columns:
        out["region_type"] = np.where(out["site"].str.contains("train|region", case=False, regex=True),
                                      "train", "site")

    out["central_ca"] = out["lat"].between(CENTRAL_LAT_MIN, CENTRAL_LAT_MAX, inclusive="both")

    if "sig" not in out.columns:
        if {"ci_lo", "ci_hi"}.issubset(out.columns):
            out["sig"] = out["ci_lo"] > 0.5
        else:
            out["sig"] = out["auc"] > 0.5

    if {"ci_lo", "ci_hi"}.issubset(out.columns):
        auc_se = (out["ci_hi"] - out["ci_lo"]) / (2 * 1.96)
        auc_se = auc_se.where(np.isfinite(auc_se) & (auc_se > 0))
        out["auc_se"] = auc_se
        out["weight"] = 1 / np.square(out["auc_se"])
    else:
        out["auc_se"] = np.nan
        out["weight"] = 1.0

    out["weight"] = out["weight"].replace([np.inf, -np.inf], np.nan).fillna(1.0)

    if "delta_auc" not in out.columns and "auc_baseline" in out.columns:
        out["delta_auc"] = out["auc"] - out["auc_baseline"]

    out = out.sort_values("lat", ascending=False).reset_index(drop=True)
    return out

def prepare_baseline_table(df):
    if df.empty:
        return df
    required = {"site", "auc_baseline"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"BASELINE_SITE_CSV missing required columns: {sorted(missing)}")
    out = df.copy()
    out["site"] = out["site"].astype(str).str.strip()
    out["auc_baseline"] = pd.to_numeric(out["auc_baseline"], errors="coerce")
    return out

def prepare_predictions(df):
    if df.empty:
        return df
    required = {"site", "date", "onset", "prob_full"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"PREDICTIONS_CSV missing required columns: {sorted(missing)}")

    out = df.copy()
    out["site"] = out["site"].astype(str).str.strip()
    out["canonical_site"] = out["site"].replace(SITE_ALIASES)
    out["date"] = pd.to_datetime(out["date"])
    out["onset"] = pd.to_numeric(out["onset"], errors="coerce").fillna(0).astype(int)
    out["prob_full"] = pd.to_numeric(out["prob_full"], errors="coerce")
    if "prob_baseline" in out.columns:
        out["prob_baseline"] = pd.to_numeric(out["prob_baseline"], errors="coerce")
    return out.sort_values(["site", "date"]).reset_index(drop=True)

site_df = prepare_site_summary(_read_csv_from_text_or_path(SITE_SUMMARY_CSV, SITE_SUMMARY_PATH, "SITE_SUMMARY"))
baseline_df = prepare_baseline_table(_read_csv_from_text_or_path(BASELINE_SITE_CSV, BASELINE_SITE_PATH, "BASELINE_SITE"))
pred_df = prepare_predictions(_read_csv_from_text_or_path(PREDICTIONS_CSV, PREDICTIONS_PATH, "PREDICTIONS"))

if not baseline_df.empty and "auc_baseline" not in site_df.columns:
    site_df = site_df.merge(baseline_df, on="site", how="left")
    site_df["delta_auc"] = site_df["auc"] - site_df["auc_baseline"]

display_cols = ["site", "lat", "lon", "auc"]
for extra in ["ci_lo", "ci_hi", "n_onset", "region_type", "sig", "auc_baseline", "delta_auc"]:
    if extra in site_df.columns:
        display_cols.append(extra)

print(f"Site summary rows: {len(site_df)}")
print(f"Predictions rows:  {len(pred_df)}")
display(site_df[display_cols].head(20))



## 3) Statistics helpers

This section is intentionally conservative:
- **weighted means** if confidence intervals are available
- **permutation test** for the Central-vs-rest contrast
- **sign test** for “more sites above random than expected by chance”
- **weighted quadratic fit** for the geographic gradient
- **leave-one-site-out sensitivity** so the pattern is not driven by one famous site


In [ ]:

def weighted_mean(x, w):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    return np.sum(w * x) / np.sum(w)

def weighted_r2(y, yhat, w):
    y = np.asarray(y, dtype=float)
    yhat = np.asarray(yhat, dtype=float)
    w = np.asarray(w, dtype=float)
    ybar = weighted_mean(y, w)
    ss_res = np.sum(w * (y - yhat) ** 2)
    ss_tot = np.sum(w * (y - ybar) ** 2)
    return 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

def weighted_polyfit(x, y, w, deg=2):
    coef = np.polyfit(
        np.asarray(x, dtype=float),
        np.asarray(y, dtype=float),
        deg=deg,
        w=np.sqrt(np.asarray(w, dtype=float))
    )
    return coef, np.poly1d(coef)

def bootstrap_mean(values, weights=None, B=N_BOOT, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    values = np.asarray(values, dtype=float)
    if weights is None:
        weights = np.ones_like(values, dtype=float)
    else:
        weights = np.asarray(weights, dtype=float)
    n = len(values)
    stats = []
    for _ in range(B):
        idx = rng.choice(np.arange(n), size=n, replace=True)
        stats.append(weighted_mean(values[idx], weights[idx]))
    stats = np.asarray(stats)
    return np.quantile(stats, [0.025, 0.975])

def permutation_weighted_group_diff(df, B=N_PERM, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    y = df["auc"].to_numpy(float)
    w = df["weight"].to_numpy(float)
    g = df["central_ca"].to_numpy(bool)

    obs = weighted_mean(y[g], w[g]) - weighted_mean(y[~g], w[~g])
    perm = np.empty(B, dtype=float)

    for i in range(B):
        gp = rng.permutation(g)
        perm[i] = weighted_mean(y[gp], w[gp]) - weighted_mean(y[~gp], w[~gp])

    p = (np.sum(perm >= obs) + 1) / (B + 1)
    return obs, p, perm

def quadratic_gradient_test(df, B=N_PERM, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    x = df["lat"].to_numpy(float)
    y = df["auc"].to_numpy(float)
    w = df["weight"].to_numpy(float)

    coef_quad, poly_quad = weighted_polyfit(x, y, w, deg=2)
    coef_lin, poly_lin = weighted_polyfit(x, y, w, deg=1)

    r2_quad = weighted_r2(y, poly_quad(x), w)
    r2_lin = weighted_r2(y, poly_lin(x), w)
    delta_r2 = r2_quad - r2_lin

    perm = np.empty(B, dtype=float)
    for i in range(B):
        yp = rng.permutation(y)
        _, p_quad = weighted_polyfit(x, yp, w, deg=2)
        _, p_lin = weighted_polyfit(x, yp, w, deg=1)
        perm[i] = weighted_r2(yp, p_quad(x), w) - weighted_r2(yp, p_lin(x), w)

    p = (np.sum(perm >= delta_r2) + 1) / (B + 1)
    peak_lat = -coef_quad[1] / (2 * coef_quad[0]) if coef_quad[0] != 0 else np.nan

    return {
        "coef_quad": coef_quad,
        "poly_quad": poly_quad,
        "r2_quad": r2_quad,
        "r2_lin": r2_lin,
        "delta_r2": delta_r2,
        "perm_p": p,
        "peak_lat": peak_lat,
        "perm_dist": perm,
    }

def bootstrap_quadratic_band(df, x_grid, B=N_BOOT, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    idx_all = np.arange(len(df))
    curves = []

    for _ in range(B):
        idx = rng.choice(idx_all, size=len(df), replace=True)
        tmp = df.iloc[idx]
        try:
            _, poly = weighted_polyfit(tmp["lat"], tmp["auc"], tmp["weight"], deg=2)
            curves.append(poly(x_grid))
        except Exception:
            continue

    curves = np.vstack(curves)
    lo = np.quantile(curves, 0.025, axis=0)
    hi = np.quantile(curves, 0.975, axis=0)
    return lo, hi

def leave_one_site_out_effect(df):
    rows = []
    for i in range(len(df)):
        tmp = df.drop(df.index[i])
        if tmp["central_ca"].sum() == 0 or (~tmp["central_ca"]).sum() == 0:
            continue
        effect = (
            weighted_mean(tmp.loc[tmp["central_ca"], "auc"], tmp.loc[tmp["central_ca"], "weight"])
            - weighted_mean(tmp.loc[~tmp["central_ca"], "auc"], tmp.loc[~tmp["central_ca"], "weight"])
        )
        rows.append({"omitted_site": df.iloc[i]["site"], "central_minus_rest_auc": effect})
    return pd.DataFrame(rows).sort_values("central_minus_rest_auc").reset_index(drop=True)

def paired_signflip_test(delta, B=N_PERM, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    delta = np.asarray(delta, dtype=float)
    obs = delta.mean()
    perm = np.empty(B, dtype=float)
    for i in range(B):
        signs = rng.choice([-1, 1], size=len(delta), replace=True)
        perm[i] = np.mean(delta * signs)
    p = (np.sum(perm >= obs) + 1) / (B + 1)
    return obs, p, perm

def summarize_site_results(df):
    out = {}

    out["n_sites"] = len(df)
    out["n_central"] = int(df["central_ca"].sum())
    out["n_noncentral"] = int((~df["central_ca"]).sum())

    out["weighted_mean_auc"] = weighted_mean(df["auc"], df["weight"])
    out["weighted_mean_ci"] = bootstrap_mean(df["auc"], df["weight"])

    above_random = int((df["auc"] > 0.5).sum())
    valid_for_sign = int((df["auc"] != 0.5).sum())
    out["n_above_random"] = above_random
    out["sign_p"] = binomtest(above_random, valid_for_sign, p=0.5, alternative="greater").pvalue

    central = df[df["central_ca"]]
    noncentral = df[~df["central_ca"]]
    out["central_weighted_mean_auc"] = weighted_mean(central["auc"], central["weight"])
    out["noncentral_weighted_mean_auc"] = weighted_mean(noncentral["auc"], noncentral["weight"])
    out["central_diff_ci"] = bootstrap_mean(
        np.r_[
            central["auc"].to_numpy(),
            -noncentral["auc"].to_numpy()
        ],
        np.r_[
            central["weight"].to_numpy(),
            noncentral["weight"].to_numpy()
        ]
    ) if False else None  # placeholder to keep summary compact

    mwu = mannwhitneyu(central["auc"], noncentral["auc"], alternative="greater")
    out["mwu_p"] = mwu.pvalue

    diff, perm_p, perm_dist = permutation_weighted_group_diff(df)
    out["central_minus_rest_weighted_diff"] = diff
    out["central_perm_p"] = perm_p
    out["central_perm_dist"] = perm_dist

    q = quadratic_gradient_test(df)
    out.update(q)

    loo = leave_one_site_out_effect(df)
    out["loo_effects"] = loo
    out["loo_min"] = loo["central_minus_rest_auc"].min()
    out["loo_max"] = loo["central_minus_rest_auc"].max()

    return out

site_stats = summarize_site_results(site_df)

print("=" * 72)
print("SITE-LEVEL EVIDENCE SYNTHESIS")
print("=" * 72)
print(f"Sites analysed:                     {site_stats['n_sites']}")
print(f"Central CA sites:                  {site_stats['n_central']}")
print(f"Non-central sites:                 {site_stats['n_noncentral']}")
print(f"Weighted mean site AUC:            {site_stats['weighted_mean_auc']:.3f}"
      f"  CI [{site_stats['weighted_mean_ci'][0]:.3f}, {site_stats['weighted_mean_ci'][1]:.3f}]")
print(f"Sites above AUC=0.5:               {site_stats['n_above_random']}/{site_stats['n_sites']}"
      f"  sign-test p={site_stats['sign_p']:.4f}")
print(f"Central CA weighted mean AUC:      {site_stats['central_weighted_mean_auc']:.3f}")
print(f"Non-central weighted mean AUC:     {site_stats['noncentral_weighted_mean_auc']:.3f}")
print(f"Central - rest weighted diff:      {site_stats['central_minus_rest_weighted_diff']:.3f}"
      f"  perm-p={site_stats['central_perm_p']:.4f}")
print(f"Central - rest MWU p-value:        {site_stats['mwu_p']:.4f}")
print(f"Weighted quadratic R²:             {site_stats['r2_quad']:.3f}")
print(f"Weighted linear R²:                {site_stats['r2_lin']:.3f}")
print(f"Quadratic improvement ΔR²:         {site_stats['delta_r2']:.3f}"
      f"  perm-p={site_stats['perm_p']:.4f}")
print(f"Estimated latitude of peak skill:  {site_stats['peak_lat']:.2f}°N")
print(f"LOO central advantage range:       [{site_stats['loo_min']:.3f}, {site_stats['loo_max']:.3f}]")



## 4) Main figure: California map + forest plot + geographic trend + sensitivity

This is the strongest presentation panel in the notebook.
It is compact enough for a talk, but still statistically honest.


In [ ]:

def plot_california_map(ax, df, value_col="auc", title="California map", cmap="viridis"):
    ax.set_facecolor("#d9edf7")

    # Simple land polygon east of the coastline
    land_lon = CA_COAST_LON + [-114.0, -114.0, CA_COAST_LON[0]]
    land_lat = CA_COAST_LAT + [CA_COAST_LAT[-1], CA_COAST_LAT[0], CA_COAST_LAT[0]]
    ax.fill(land_lon, land_lat, color="#efe6d8", zorder=0)
    ax.plot(CA_COAST_LON, CA_COAST_LAT, color="black", lw=1.1, zorder=1)

    ax.axhspan(CENTRAL_LAT_MIN, CENTRAL_LAT_MAX, color="gold", alpha=0.14, zorder=0)
    ax.axhline(CENTRAL_LAT_MIN, color="darkgoldenrod", lw=0.8, ls="--", alpha=0.7)
    ax.axhline(CENTRAL_LAT_MAX, color="darkgoldenrod", lw=0.8, ls="--", alpha=0.7)

    size = 55 + 18 * df.get("n_onset", pd.Series(np.repeat(4, len(df)))).fillna(4).to_numpy()
    edge_lw = np.where(df["sig"].astype(bool), 1.8, 0.7)

    sc = ax.scatter(
        df["lon"], df["lat"],
        c=df[value_col],
        s=size,
        cmap=cmap,
        edgecolor="black",
        linewidth=edge_lw,
        zorder=3
    )

    label_mask = df["sig"].astype(bool) | df["site"].str.contains("train|Region", case=False, regex=True)
    for _, row in df[label_mask].iterrows():
        label = row["site"].replace(" (train)", "").replace(" Region", "")
        ax.text(row["lon"] + 0.10, row["lat"], label, fontsize=7.2, va="center", zorder=4)

    ax.set_xlim(-125.4, -116.2)
    ax.set_ylim(32.2, 42.2)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title(title)
    return sc

fig = plt.figure(figsize=(15, 11))
gs = fig.add_gridspec(2, 2, width_ratios=[1.0, 1.0], height_ratios=[1.0, 1.0], hspace=0.32, wspace=0.25)

# A) Map
ax_map = fig.add_subplot(gs[:, 0])
sc = plot_california_map(ax_map, site_df, value_col="auc", title="A. Held-out site performance across California")
cbar = fig.colorbar(sc, ax=ax_map, fraction=0.046, pad=0.04)
cbar.set_label("Site AUC")

legend_elements = [
    mpatches.Patch(facecolor="gold", alpha=0.14, edgecolor="none", label="Central CA upwelling core"),
]
ax_map.legend(handles=legend_elements, loc="lower right", frameon=True, fontsize=9)

# B) Forest plot
ax_forest = fig.add_subplot(gs[0, 1])
forest_df = site_df.sort_values("lat", ascending=True).reset_index(drop=True)
y = np.arange(len(forest_df))

if {"ci_lo", "ci_hi"}.issubset(forest_df.columns):
    xerr = np.vstack([
        forest_df["auc"] - forest_df["ci_lo"],
        forest_df["ci_hi"] - forest_df["auc"],
    ])
    ax_forest.errorbar(
        forest_df["auc"], y,
        xerr=xerr,
        fmt="o",
        color="black",
        ecolor="gray",
        elinewidth=1.2,
        capsize=3,
        markersize=5
    )
else:
    ax_forest.scatter(forest_df["auc"], y, color="black", s=28)

ax_forest.axvline(0.5, color="firebrick", lw=1.2, ls="--", label="Random AUC = 0.5")
ax_forest.set_yticks(y)
ax_forest.set_yticklabels(forest_df["site"])
ax_forest.set_xlabel("AUC")
ax_forest.set_title("B. Forest plot with confidence intervals")
ax_forest.set_xlim(max(0.0, forest_df["auc"].min() - 0.08), min(1.02, forest_df["auc"].max() + 0.10))
ax_forest.legend(loc="lower right", fontsize=9)

# C) Latitude trend
ax_trend = fig.add_subplot(gs[1, 1])

x = site_df["lat"].to_numpy(float)
y = site_df["auc"].to_numpy(float)
w = site_df["weight"].to_numpy(float)

x_grid = np.linspace(x.min() - 0.25, x.max() + 0.25, 300)
trend = site_stats["poly_quad"](x_grid)
band_lo, band_hi = bootstrap_quadratic_band(site_df, x_grid)

scatter_size = 35 + 18 * site_df.get("n_onset", pd.Series(np.repeat(4, len(site_df)))).fillna(4).to_numpy()
ax_trend.scatter(x, y, s=scatter_size, color="black", alpha=0.75)
ax_trend.plot(x_grid, trend, lw=2.2, color="tab:blue", label="Weighted quadratic fit")
ax_trend.fill_between(x_grid, band_lo, band_hi, color="tab:blue", alpha=0.16, label="Bootstrap 95% band")
ax_trend.axhline(0.5, color="firebrick", lw=1.0, ls="--")
ax_trend.axvspan(CENTRAL_LAT_MIN, CENTRAL_LAT_MAX, color="gold", alpha=0.12)
ax_trend.set_xlabel("Latitude")
ax_trend.set_ylabel("AUC")
ax_trend.set_title("C. Geographic gradient in held-out performance")
ax_trend.legend(loc="upper left", fontsize=9)
ax_trend.text(
    0.02, 0.03,
    f"Peak latitude = {site_stats['peak_lat']:.2f}°N\n"
    f"Quadratic ΔR² = {site_stats['delta_r2']:.3f}, perm-p = {site_stats['perm_p']:.4f}",
    transform=ax_trend.transAxes,
    ha="left", va="bottom",
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.85, edgecolor="0.7")
)

# D) Leave-one-site-out sensitivity inset
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
ax_inset = inset_axes(ax_trend, width="46%", height="56%", loc="upper right", borderpad=1.0)
loo_df = site_stats["loo_effects"]
ax_inset.hlines(np.arange(len(loo_df)), xmin=0, xmax=loo_df["central_minus_rest_auc"], color="0.7")
ax_inset.plot(loo_df["central_minus_rest_auc"], np.arange(len(loo_df)), "o", color="tab:green", markersize=4)
ax_inset.axvline(0, color="firebrick", lw=1.0, ls="--")
ax_inset.set_yticks([])
ax_inset.set_xlabel("LOO effect", fontsize=8)
ax_inset.set_title("Sensitivity", fontsize=9)

fig.suptitle("Spatial evidence synthesis of held-out kelp early-warning performance", fontsize=15, y=0.985)
fig.savefig(OUTPUT_DIR / "spatial_evidence_main_figure.png", bbox_inches="tight")
plt.show()

print(f"Saved: {OUTPUT_DIR / 'spatial_evidence_main_figure.png'}")



## 5) Optional: full model vs baseline by site

This section only runs if `auc_baseline` is available.  
It is particularly useful if your baseline is something like:
- SST-only
- SST + upwelling without EWS
- persistence-only

That lets you make the strongest claim:
**“EWS adds value beyond climate-only prediction, and the gain is spatially structured.”**


In [ ]:

if "auc_baseline" in site_df.columns and site_df["auc_baseline"].notna().sum() >= 3:
    compare_df = site_df.dropna(subset=["auc", "auc_baseline"]).copy()
    compare_df["delta_auc"] = compare_df["auc"] - compare_df["auc_baseline"]

    delta = compare_df["delta_auc"].to_numpy(float)
    obs_delta, perm_p_delta, _ = paired_signflip_test(delta)
    try:
        wilcox_p = wilcoxon(delta, alternative="greater").pvalue
    except Exception:
        wilcox_p = np.nan

    print("=" * 72)
    print("FULL MODEL VS BASELINE")
    print("=" * 72)
    print(f"Sites compared:                {len(compare_df)}")
    print(f"Mean ΔAUC (full - baseline):   {delta.mean():.3f}")
    print(f"Median ΔAUC:                   {np.median(delta):.3f}")
    print(f"Sign-flip permutation p:       {perm_p_delta:.4f}")
    print(f"Wilcoxon p-value:              {wilcox_p:.4f}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={"width_ratios": [1.1, 1.0]})

    # Paired slope plot
    ax = axes[0]
    comp = compare_df.sort_values("delta_auc", ascending=False).reset_index(drop=True)
    for i, row in comp.iterrows():
        ax.plot([0, 1], [row["auc_baseline"], row["auc"]], color="0.6", lw=1.2)
        ax.scatter([0, 1], [row["auc_baseline"], row["auc"]], color=["tab:gray", "tab:blue"], s=36)
        ax.text(1.03, row["auc"], row["site"], fontsize=8, va="center")
    ax.axhline(0.5, color="firebrick", lw=1.0, ls="--")
    ax.set_xlim(-0.2, 1.45)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Baseline", "Full model"])
    ax.set_ylabel("AUC")
    ax.set_title("A. Site-wise improvement over baseline")

    # California delta map
    ax = axes[1]
    sc2 = plot_california_map(ax, compare_df, value_col="delta_auc", title="B. Where EWS adds the most", cmap="coolwarm")
    cbar2 = fig.colorbar(sc2, ax=ax, fraction=0.046, pad=0.04)
    cbar2.set_label("ΔAUC (full - baseline)")

    fig.suptitle("Added value of the full early-warning model over baseline", fontsize=14, y=0.98)
    fig.savefig(OUTPUT_DIR / "spatial_evidence_baseline_comparison.png", bbox_inches="tight")
    plt.show()

    print(f"Saved: {OUTPUT_DIR / 'spatial_evidence_baseline_comparison.png'}")
else:
    print("Baseline comparison skipped: add auc_baseline to SITE_SUMMARY_CSV or paste BASELINE_SITE_CSV.")



## 6) Optional: calibration and horizon-aware utility

This section is for when you paste quarter-level **out-of-fold / held-out predictions**.

It gives you something most student projects do not have:
- **reliability / calibration**
- warning quality for **1-quarter, 2-quarter, and 4-quarter horizons**
- a clean way to say whether the model is just scoring risk or actually giving useful warning time

Important: this section is intentionally **optional** so the notebook stays simple if you only want the spatial synthesis.


In [ ]:

RISK_THRESHOLD = 0.35  # change only if you used a pre-specified threshold elsewhere

def add_future_event_labels(df, horizons=(1, 2, 4)):
    out = df.copy().sort_values(["site", "date"]).reset_index(drop=True)
    for h in horizons:
        future_col = f"event_within_{h}q"
        out[future_col] = 0
        for site, idx in out.groupby("site").groups.items():
            g = out.loc[list(idx)].sort_values("date").copy()
            y = g["onset"].to_numpy(int)
            future = np.zeros(len(g), dtype=int)
            for i in range(len(g)):
                upper = min(len(g), i + h + 1)
                future[i] = int(np.any(y[i + 1:upper] == 1))
            out.loc[g.index, future_col] = future
    return out

def safe_metric(metric_fn, y_true, y_score):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    if len(np.unique(y_true)) < 2:
        return np.nan
    return metric_fn(y_true, y_score)

if not pred_df.empty:
    pred_eval = add_future_event_labels(pred_df, horizons=(1, 2, 4))

    print("=" * 72)
    print("CALIBRATION + HORIZON-AWARE UTILITY")
    print("=" * 72)

    rows = []
    for h in (1, 2, 4):
        y = pred_eval[f"event_within_{h}q"].to_numpy(int)
        p = pred_eval["prob_full"].to_numpy(float)
        alarm = (p >= RISK_THRESHOLD).astype(int)

        tp = int(np.sum((alarm == 1) & (y == 1)))
        fp = int(np.sum((alarm == 1) & (y == 0)))
        fn = int(np.sum((alarm == 0) & (y == 1)))

        precision = tp / (tp + fp) if (tp + fp) else np.nan
        recall = tp / (tp + fn) if (tp + fn) else np.nan
        alarm_rate = alarm.mean()

        rows.append({
            "horizon_q": h,
            "prevalence": y.mean(),
            "auc": safe_metric(roc_auc_score, y, p),
            "ap": safe_metric(average_precision_score, y, p),
            "brier": safe_metric(brier_score_loss, y, p),
            "precision_at_threshold": precision,
            "recall_at_threshold": recall,
            "alarm_rate": alarm_rate,
        })

    horizon_df = pd.DataFrame(rows)
    display(horizon_df)

    # Calibration on 1-quarter onset for comparability
    y1 = pred_eval["event_within_1q"].to_numpy(int)
    p1 = pred_eval["prob_full"].to_numpy(float)

    if len(np.unique(y1)) >= 2:
        frac_pos, mean_pred = calibration_curve(y1, p1, n_bins=8, strategy="quantile")

        fig, axes = plt.subplots(1, 2, figsize=(12, 5))

        # Calibration plot
        ax = axes[0]
        ax.plot([0, 1], [0, 1], color="0.5", ls="--", lw=1.0)
        ax.plot(mean_pred, frac_pos, marker="o", lw=2.0, color="tab:blue")
        ax.set_xlabel("Predicted probability")
        ax.set_ylabel("Observed event frequency")
        ax.set_title("A. Calibration for next-quarter onset")

        # Horizon utility plot
        ax = axes[1]
        ax.plot(horizon_df["horizon_q"], horizon_df["auc"], marker="o", lw=2.0, label="AUC")
        ax.plot(horizon_df["horizon_q"], horizon_df["recall_at_threshold"], marker="o", lw=2.0, label=f"Recall @ {RISK_THRESHOLD:.2f}")
        ax.plot(horizon_df["horizon_q"], horizon_df["precision_at_threshold"], marker="o", lw=2.0, label=f"Precision @ {RISK_THRESHOLD:.2f}")
        ax.plot(horizon_df["horizon_q"], horizon_df["alarm_rate"], marker="o", lw=2.0, label="Alarm rate")
        ax.set_xlabel("Warning horizon (quarters)")
        ax.set_ylabel("Metric value")
        ax.set_xticks([1, 2, 4])
        ax.set_title("B. Warning quality across management horizons")
        ax.legend(loc="best", fontsize=8)

        fig.suptitle("Calibration and actionability of held-out risk predictions", fontsize=14, y=1.02)
        fig.savefig(OUTPUT_DIR / "spatial_evidence_calibration_horizons.png", bbox_inches="tight")
        plt.show()

        print(f"Saved: {OUTPUT_DIR / 'spatial_evidence_calibration_horizons.png'}")
    else:
        print("Calibration skipped: the pasted predictions do not contain both classes for next-quarter onset.")
else:
    print("Prediction-level utility skipped: paste PREDICTIONS_CSV later if you want calibration + horizon metrics.")



## 7) Auto-generated board / slide language

This final cell gives you clean language you can adapt for:
- poster captions
- slide notes
- judge Q&A


In [ ]:

summary_lines = [
    "ISEF-ready summary",
    "-" * 72,
    f"Held-out sites analysed: {site_stats['n_sites']}",
    (
        f"Weighted mean site AUC was {site_stats['weighted_mean_auc']:.3f} "
        f"(bootstrap 95% CI {site_stats['weighted_mean_ci'][0]:.3f} to {site_stats['weighted_mean_ci'][1]:.3f})."
    ),
    (
        f"{site_stats['n_above_random']} of {site_stats['n_sites']} sites exceeded random prediction "
        f"(sign-test p={site_stats['sign_p']:.4f})."
    ),
    (
        f"Central California showed higher held-out skill than the rest of the coast "
        f"(weighted ΔAUC={site_stats['central_minus_rest_weighted_diff']:.3f}, "
        f"permutation p={site_stats['central_perm_p']:.4f})."
    ),
    (
        f"The latitude-performance relationship was better explained by a weighted quadratic curve "
        f"than by a straight line (ΔR²={site_stats['delta_r2']:.3f}, permutation p={site_stats['perm_p']:.4f}), "
        f"with peak predicted skill near {site_stats['peak_lat']:.2f}°N."
    ),
    (
        f"Leave-one-site-out sensitivity kept the Central-California advantage positive in every omission "
        f"(range {site_stats['loo_min']:.3f} to {site_stats['loo_max']:.3f}), "
        "showing that the pattern was not driven by a single site."
    ),
]

if "auc_baseline" in site_df.columns and site_df["auc_baseline"].notna().sum() >= 3:
    compare_df = site_df.dropna(subset=["auc", "auc_baseline"]).copy()
    compare_df["delta_auc"] = compare_df["auc"] - compare_df["auc_baseline"]
    summary_lines.append(
        f"Against the pasted baseline, the full model improved site-level AUC by a mean of "
        f"{compare_df['delta_auc'].mean():.3f}."
    )

summary_text = "\n".join(summary_lines)
print(summary_text)

(OUTPUT_DIR / "isef_ready_summary.txt").write_text(summary_text)
site_df.to_csv(OUTPUT_DIR / "cleaned_site_summary.csv", index=False)

print()
print(f"Saved summary text:  {OUTPUT_DIR / 'isef_ready_summary.txt'}")
print(f"Saved cleaned table: {OUTPUT_DIR / 'cleaned_site_summary.csv'}")
